In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Orchestrating Multi-Agent Systems with Google Agent Development Kit

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/iamthuya/google-cloud-workshops/blob/main/ai-agents/agent-development-kit/orchestrating_multi_agent_systems.ipynb">
      <img src="https://avatars.githubusercontent.com/u/33467679?s=200&v=4" width="30px" alt="Google Colaboratory logo"><br> Run in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fiamthuya%2Fgoogle-cloud-workshops%2Fmain%2Fai-agents%2Fagent-development-kit%2Forchestrating_multi_agent_systems.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Run in Colab Enterprise
    </a>
  </td>      
  <td style="text-align: center">
    <a href="https://github.com/iamthuya/google-cloud-workshops/blob/main/ai-agents/agent-development-kit/orchestrating_multi_agent_systems.ipynb">
      <img src="https://github.blog/wp-content/uploads/2013/04/074d0b06-a5e3-11e2-8b7f-9f09eb2ddfae.jpg" width="55px" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/iamthuya/google-cloud-workshops/blob/main/ai-agents/agent-development-kit/orchestrating_multi-agent_systems.ipynb">
      <img src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
</table>

| | |
|-|-|
|Author(s) | [Thu Ya Kyaw](https://github.com/iamthuya) |

## Environment Setup

### Install Necessary Libraries

Here you will install required Python packages for this lab.

In [ ]:
%pip install -U -q google-adk

In [ ]:
# Check ADK version
from google import adk

print(adk.__version__)

### Restart current runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which will restart the current kernel.

Don't worry if you see a notification message like `Your session crashed for an unknown reason.` This is expected as you are shutting down the kernel from the same instance using code.


In [ ]:
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

<div class="alert alert-block alert-warning">
<b>⚠️ The kernel is going to restart. Please wait until it is finished before continuing to the next step. ⚠️</b>
</div>

### Import Required Libraries

In [ ]:
import os
import asyncio

from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, ToolContext
from google.genai import types

from IPython.display import Markdown, display

### Define Environment Variables

In [ ]:
# Check if you are running this notebook from google colab
try:
  from google.colab import userdata
  os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
  os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
  print("*** Using Google AI Backend ***")

except:
  os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
  os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

  if os.getenv("GOOGLE_CLOUD_PROJECT") is None:  # check if project-id is already assigned
    project_id = "replace-with-your-project-id"  #@param {type:"string"}
    os.environ["GOOGLE_CLOUD_PROJECT"] = project_id

  import sys

  if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()


  print(("*** Using Vertex AI Backend ***\n"
         f"PROJECT: {os.environ['GOOGLE_CLOUD_PROJECT']}\n"
         f"LOCATION: {os.environ['GOOGLE_CLOUD_LOCATION']}"
       ))


## Sequential Agents

In this section, you'll create an agent that researches a topic, then writes in detail about it, and finally edits the output. These steps must happen in order: research first, then writing, and then editing, because you can't write without research, and you can't edit without a written piece.

In [ ]:
# Define variables
GEMINI_MODEL_NAME = "gemini-2.5-flash"
APP_NAME = "sequential_app"
USER_ID = "sequential_user"
SESSION_ID = "sequential_session"

#### Initialize the agents and pipeline

In [ ]:
# Define respective agents and pipeline
researcher = LlmAgent(
    name="ResearcherAgent",
    model=GEMINI_MODEL_NAME,
    instruction="Your task is to use 'google_search' tool to find all the relavent information about a given topic and write a comprehensive report about that topic.",
    tools=[google_search],
    description="Researches about a topic",
    output_key="research_results"
)

writer = LlmAgent(
    name="WriterAgent",
    model=GEMINI_MODEL_NAME,
    instruction="Your task is to write a comprehensive report using this information: {research_results}. Make sure to expend on all the topics from the provided information.",
    description="Write about a comprehensive report based on the provided information",
    output_key="comprehensive_report"
)

editor = LlmAgent(
    name="EditorAgent",
    model=GEMINI_MODEL_NAME,
    instruction="Your task is to edit a report: {comprehensive_report}. Perform a quality check on the text and improve it if necessary.",
    description="Perform quality check on a written report",
)

sequential_pipeline = SequentialAgent(
    name="ContentPipelineAgent",
    sub_agents=[researcher, writer, editor],
    description="Executes a sequence of researching, writing, and editing.",
)

### Initialize Session & Runner

In [ ]:
# Initialize Session
session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# Initialize Runner
sequential_runner = Runner(
    agent=sequential_pipeline,
    app_name=APP_NAME,
    session_service=session_service,
)

### Define a function to interact with Agent

In [ ]:
# Define a function to interact with the agent
async def run_agent(runner, user_id, session_id, user_query):
    user_message = types.Content(
        role="user",
        parts=[types.Part(text=user_query)]
    )

    events = runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=user_message
    )

    output = {}
    async for event in events:
        if event.is_final_response():
            author = event.author
            report = event.content.parts[0].text
            output[author] = report

            display(Markdown(f"## {author}"))
            display(Markdown(f"{report}"))

    return output

In [ ]:
topic = "Write about the current state of generative ai"
output = await run_agent(sequential_runner, USER_ID, SESSION_ID, topic)
report = output['EditorAgent']

## Parallel Agents

Sometimes, running agents in parallel is desirable, particularly for tasks like translation. The Agent Developer Kit supports such parallel orchestration. In this session, you will create a parallel pipeline designed to translate a report into several languages simultaneously.

In [ ]:
# Define variables
APP_NAME = "parallel_app"
USER_ID = "parallel_user"
SESSION_ID = "parallel_session"

#### Initialize the agents and pipeline

In [ ]:
# Define respective agents and pipeline

kannada_translator = LlmAgent(
    name="KannadaAgent",
    model=GEMINI_MODEL_NAME,
    instruction="Your task is to translate the given report into Kannada lanaguage",
    description="Translate a given report to Kannada",
    output_key="kannada_version"
)

thai_translator = LlmAgent(
    name="ThaiAgent",
    model=GEMINI_MODEL_NAME,
    instruction="Your task is to translate the given report into Thai lanaguage",
    description="Translate a given report to Thai",
    output_key="thai_version"
)

parallel_pipeline = ParallelAgent(
    name="TranslationPipelineAgent",
    sub_agents=[kannada_translator, thai_translator],
    description="Runs multiple tranlsation agents in parallel to translate the report.",
)

#### Initialize Session & Runner

In [ ]:
# Initialize Session
session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# Initialize Runner
parallel_runner = Runner(
    agent=parallel_pipeline,
    app_name=APP_NAME,
    session_service=session_service,
)

In [ ]:
prompt = f"translate this report: {report}"
output = await run_agent(parallel_runner, USER_ID, SESSION_ID, prompt)

## Loop Agents

Some workflow involves repetition or iterative refinement, such as like revising code. ADK also support such workflows.



In [ ]:
# Declare constants
APP_NAME = "loop_app" # New App Name
USER_ID = "loop_user"
SESSION_ID = "loop_session" # New Base Session ID
STATE_INITIAL_TOPIC = "Kopi Luwak"

# Define state variables
STATE_CURRENT_DOC = "current_document"
STATE_CRITICISM = "criticism"

# Define the exact phrase the Critic should use to signal completion
COMPLETION_PHRASE = "No major issues found."

#### Tools

In [ ]:
# Define exit loop tool
def exit_loop(tool_context: ToolContext):
    """Call this function ONLY when the critique indicates no further changes are needed, signaling the iterative process should end."""
    print(f"[Tool Call] exit_loop triggered by {tool_context.agent_name}")
    tool_context.actions.escalate = True
    # Return empty dict as tools should typically return JSON-serializable output
    return {}

#### Initial Writer Agent

In [ ]:
# STEP 1: Initial Writer Agent (Runs ONCE at the beginning)
initial_writer_agent = LlmAgent(
    name="InitialWriterAgent",
    model=GEMINI_MODEL_NAME,
    include_contents='none',
    instruction=f"""You are a Creative Writing Assistant tasked with starting a story.
    Write the *first draft* of a short story (aim for 2-4 sentences).
    Base the content *only* on the topic provided below. Try to introduce a specific element (like a character, a setting detail, or a starting action) to make it engaging.
    Topic: {{initial_topic}}

    Output *only* the story/document text. Do not add introductions or explanations.
""",
    description="Writes the initial document draft based on the topic, aiming for some initial substance.",
    output_key=STATE_CURRENT_DOC
)

#### Critic Agent

In [ ]:
# STEP 2a: Critic Agent (Inside the Refinement Loop)
critic_agent_in_loop = LlmAgent(
    name="CriticAgent",
    model=GEMINI_MODEL_NAME,
    include_contents='none',
    # MODIFIED Instruction: More nuanced completion criteria, look for clear improvement paths.
    instruction=f"""You are a Constructive Critic AI reviewing a short document draft (typically 2-6 sentences). Your goal is balanced feedback.

    **Document to Review:**
    ```
    {{current_document}}
    ```

    **Task:**
    Review the document for clarity, engagement, and basic coherence according to the initial topic (if known).

    IF you identify 1-2 *clear and actionable* ways the document could be improved to better capture the topic or enhance reader engagement (e.g., "Needs a stronger opening sentence", "Clarify the character's goal"):
    Provide these specific suggestions concisely. Output *only* the critique text.

    ELSE IF the document is coherent, addresses the topic adequately for its length (should be 1 page long), and has no glaring errors or obvious omissions:
    Respond *exactly* with the phrase "{COMPLETION_PHRASE}" and nothing else. It doesn't need to be perfect, just functionally complete for this stage. Avoid suggesting purely subjective stylistic preferences if the core is sound.

    Do not add explanations. Output only the critique OR the exact completion phrase.
""",
    description="Reviews the current draft, providing critique if clear improvements are needed, otherwise signals completion.",
    output_key=STATE_CRITICISM
)

### Refine Agent

In [ ]:
# STEP 2b: Refiner/Exiter Agent (Inside the Refinement Loop)
refiner_agent_in_loop = LlmAgent(
    name="RefinerAgent",
    model=GEMINI_MODEL_NAME,
    # Relies solely on state via placeholders
    include_contents='none',
    instruction=f"""You are a Creative Writing Assistant refining a document based on feedback OR exiting the process.
    **Current Document:**
    ```
    {{current_document}}
    ```
    **Critique/Suggestions:**
    {{criticism}}

    **Task:**
    Analyze the 'Critique/Suggestions'.
    IF the critique is *exactly* "{COMPLETION_PHRASE}":
    You MUST call the 'exit_loop' function. Do not output any text.
    ELSE (the critique contains actionable feedback):
    Carefully apply the suggestions to improve the 'Current Document'. Output *only* the refined document text.

    Do not add explanations. Either output the refined document OR call the exit_loop function.
""",
    description="Refines the document based on critique, or calls exit_loop if critique indicates completion.",
    tools=[exit_loop], # Provide the exit_loop tool
    output_key=STATE_CURRENT_DOC # Overwrites state['current_document'] with the refined version
)

#### Refinement Loop Agent

In [ ]:
# STEP 2c: Refinement Loop Agent
refinement_loop = LoopAgent(
    name="RefinementLoop",
    # Agent order is crucial: Critique first, then Refine/Exit
    sub_agents=[
        critic_agent_in_loop,
        refiner_agent_in_loop,
    ],
    max_iterations=5 # Limit loops
)

#### Overall Sequential Agent

In [ ]:
# STEP 3: Overall Sequential Pipeline
# For ADK tools compatibility, the root agent must be named `root_agent`
root_agent = SequentialAgent(
    name="IterativeWritingPipeline",
    sub_agents=[
        initial_writer_agent, # Run first to create initial doc
        refinement_loop       # Then run the critique/refine loop
    ],
    description="Writes an initial document and then iteratively refines it with critique using an exit tool."
)

In [ ]:
# Initialize Session
session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    state={"initial_topic": "Briyani"}
)

# Initialize Runner
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

In [ ]:
await run_agent(runner, USER_ID, SESSION_ID, "write a comprehensive essay")

# Conclusions

You have explored how to create an agent using Gemini and Agent Development Kit.